### Low rank analysis - Time evolution
Import libraries

In [ ]:
import os
import sys

libs = {
  "./../modules",
  "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}
for lib in libs:
  sys.path.append(lib)

In [ ]:
import json
import numpy as np
import scipy as sp
import dask.array as da

from silx.io.dictdump import h5todict

In [ ]:
import plotting as plt_mod

from converter import Converter

Define inputs

In [ ]:
# Path to database
path_to_dtb = "./../../database/"
# Path to 0D solutions
path_to_sols = [
  "./../../datagen/runs/samples/data/train/sols.hdf5",
  "./../../datagen/runs/samples/data/valid/sols.hdf5"
]
# Path to saving
path_to_save = "./../output/lowrank_time/"

In [ ]:
# Plotting
saving = True
showing = False
# Variables to decompose
variables = {
  "n_em": r"$\tilde{n}_{e^-}$",
  "n_Ar1": r"$\tilde{n}_{Ar_1}$",
  "pe": r"$\tilde{p}_e$",
  "Te": r"$\tilde{T}_e$",
  "Th": r"$\tilde{T}_h$"
}

Define paths

In [ ]:
for k in variables.keys():
  kpath = path_to_save + f"/{k}/"
  os.makedirs(kpath, exist_ok=True)

Read solutions

In [ ]:
conv = Converter(path_to_dtb)

In [ ]:
sols = []
for p in path_to_sols:
  sols.extend(list(h5todict(p).values()))
phys = [conv.prim_to_phys(**sol) for sol in sols]

Plot solutions

In [ ]:
def get_xy(
  var,
  phys,
  qscale=None,
  shift=False
):
  # Input
  x = phys["t"].squeeze()
  if (qscale is not None):
    x = x * phys[qscale][0]
  # Output
  y = np.log(phys[var].squeeze())
  if shift:
    y = y - y[0]
  return x, y

> Function of $t$

In [ ]:
for (k, label) in variables.items():
  plt_mod.multi_traj(
    xy=[get_xy(var=k, phys=p, shift=True) for p in phys],
    labels=['$t$ [s]', label],
    scales=['log', 'linear'],
    figname=path_to_save + f"/{k}/data.png",
    save=saving,
    show=showing
  )

> Function of $t\cdot p_h(0)$

In [ ]:
for (k, label) in variables.items():
  plt_mod.multi_traj(
    xy=[get_xy(var=k, phys=p, qscale="ph", shift=True) for p in phys],
    labels=['$t$ [s]', label],
    scales=['log', 'linear'],
    figname=path_to_save + f"/{k}/data_ph.png",
    save=saving,
    show=showing
  )

> Function of $t\cdot p_e(0)$

In [ ]:
for (k, label) in variables.items():
  plt_mod.multi_traj(
    xy=[get_xy(var=k, phys=p, qscale="pe", shift=True) for p in phys],
    labels=['$t$ [s]', label],
    scales=['log', 'linear'],
    figname=path_to_save + f"/{k}/data_pe.png",
    save=saving,
    show=showing
  )

Perfoming SVD
> Interpolate solutions onto the same grid

In [ ]:
def interpolate(t, var, phys):
  x, y = get_xy(var, phys, qscale="ph", shift=False)
  # Extend domain
  x = np.concatenate([x, x[-1:]*1e2])
  y = np.concatenate([y, y[-1:]])
  # Interpolate
  return sp.interpolate.interp1d(x, y, kind='cubic')(t)

In [ ]:
tp_lim = (-3, 1)
tp = np.logspace(*tp_lim, 200)

In [ ]:
y = {}
for k in variables.keys():
  yk = [interpolate(t=tp, var=k, phys=p) for p in phys]
  y[k] = np.vstack(yk)

> SVD decomposition

In [ ]:
svd = {}
for k in variables.keys():
  # Standardize
  mean = np.mean(y[k], axis=0)
  std = np.std(y[k], axis=0)
  yk = (y[k] - mean) / std
  # Perform SVD
  yk = da.from_array(yk.T, chunks="auto")
  u, s, vh = da.linalg.svd_compressed(yk, k=100)
  s = s.compute()
  u = u.compute()
  svd[k] = {
    "mean": mean,
    "std": std,
    "s": s,
    "u": u,
  }

> Singular values decay

In [ ]:
eps = 1e-6
phi, rdim = {}, {"eps": eps}
for k in variables.keys():
  # Cumulative energy
  s = svd[k]["s"]
  cs = 1.0 / np.sum(s**2)
  cs *= np.cumsum(s**2)
  # Number of principal components
  dim = np.where(cs > 1-eps)[0][0] + 1
  rdim[k] = int(dim)
  plt_mod.cumenergy(
    cs[:dim],
    figname=path_to_save + f"/{k}/cum_en.png",
    save=saving,
    show=showing
  )
  # Extract modes
  svd[k]["phi"] = svd[k]["u"][:,:dim]
# Save number of principal components
with open(path_to_save + "/rdim.json", "w") as file:
  json.dump(rdim, file, indent=2, ensure_ascii=False)

> Test reconstruction

In [ ]:
def reconstruct(y, svd):
  y = (y - svd["mean"]) / svd["std"]
  y = svd["phi"] @ (svd["phi"].T @ y)
  return y * svd["std"] + svd["mean"]

In [ ]:
cases = np.random.randint(low=0, high=len(sols), size=10)
for (k, label) in variables.items():
  kpath = path_to_save + f"/{k}/rec/"
  os.makedirs(kpath, exist_ok=True)
  for icase in cases:
    yi = interpolate(t=tp, var=k, phys=phys[icase])
    yr = reconstruct(yi, svd[k])
    plt_mod.traj(
      x=tp,
      y_true=yi,
      y_pred=yr,
      labels=[r'$t\cdot p_{h}(0)$ [s Pa]', label],
      scales=["log", "linear"],
      filename=kpath + f"/{str(icase).zfill(6)}.png",
      save=saving,
      show=showing
    )